# Helio24 · İlk gerçek veri deneyi
**Soru:** Seçilen HARP bölgesi önümüzdeki 24 saatte M veya X sınıfı patlama üretecek mi?

12 saatlik geçmiş → 24 saatlik olasılık. Bu notebook, resmi SWAN-SF arşivlerinin etiketten bağımsız %10 örneklemini kullanır. Eğitim: bölüm 1; doğrulama: bölüm 3; test: bölüm 5. Tam veri seti sonucu değildir.

Önce proje kökünde `python scripts/download.py` çalıştırın. [Veri sözleşmesi](../docs/data-contract.md), etiket aralıklarını, kaynakları ve sınırlamaları açıklar. **Canlı tahmin veya uydu hasarı tahmini değildir.**

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "helio24").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from helio24.data import build_dataset, chronological_split, MAGNETIC_COLUMNS, FEATURE_COLUMNS
from helio24.modeling import run_experiment


## 1. Kaynak ve sınıf dağılımı
Arşivler resmi MD5 sağlama toplamlarıyla doğrulanır. Örnekleme etiketi kullanmaz. Aşağıdaki sayılar **zaman penceresi** sayısıdır, birbirinden bağımsız patlama sayısı değildir.

In [ ]:
rows = []
for partition in [1, 3, 5]:
    manifest = json.loads((ROOT / f"data/partition{partition}-manifest.json").read_text())
    for scope, field in [("Tam arşiv", "full_class_counts"), ("%10 örneklem", "sample_class_counts")]:
        rows.append({"partition": partition, "scope": scope, **manifest[field]})
distribution = pd.DataFrame(rows).fillna(0)
display(distribution)
distribution.set_index(["partition", "scope"])[["N", "B", "C", "M", "X"]].plot.bar(stacked=True, figsize=(10,4), title="Sınıf dağılımı")
plt.ylabel("Tahmin penceresi sayısı")
plt.tight_layout()
plt.show()

## 2. Veriyi aç ve kaliteyi incele
Her dosya 60 zaman adımı içerir. 24 manyetik değişkenden ortalama, standart sapma, son değer ve başlangıca göre değişim çıkarılır. Kimlikler ve sonuç sütunları modele girmez. Güvenilmeyen satırlar maskelenir; aşırı eksik pencereler ve katalog zamanıyla uyuşmayan sonuçlar elenir.

In [ ]:
# Bu hücre ham örneklemi yeniden inceler; tahmin sonuçlarını önbellekten almaz.
table, audit = build_dataset(root=ROOT)
display(pd.DataFrame([{k: v for k, v in p.items() if k not in ["missing_by_parameter", "full_class_counts", "sample_class_counts"]} for p in audit["partitions"]]))
missing = pd.DataFrame({p["partition"]: p["missing_by_parameter"] for p in audit["partitions"]})
display(missing.rename_axis("parameter"))
exclusions = pd.read_csv(ROOT / "data/prepared/exclusions.csv")
display(exclusions.groupby(["partition", "reason"]).size().rename("excluded").to_frame())
display(table.groupby(["partition", "flare_class"]).size().unstack(fill_value=0))
print(f"{len(FEATURE_COLUMNS)} özellik; sonuç, kimlik ve gelecek bilgisi girdilere dahil değil.")

## 3. Zaman sıralı ayırma ve sızıntı denetimi
Rastgele satır bölme yoktur. Sınırı aşan gözlem/sonuç aralıkları temizlenir; önceki bölümlerdeki HARP kimlikleri sonraki bölümlere taşınmaz. Eksik değer doldurma ve ölçekleme yalnızca eğitimden öğrenilir.

In [ ]:
splits, split_audit = chronological_split(table)
display(pd.DataFrame([{ "split": name, "n": len(group), "positives": int(group.target.sum()), "prevalence": group.target.mean(), "harps": group.harp.nunique(), "first_observation": group.observation_start.min(), "last_prediction_end": group.prediction_end.max()} for name, group in splits.items()]))
display(pd.DataFrame(split_audit))

## 4. Referans, lojistik regresyon ve XGBoost
Referans model her kayda eğitim olay oranını verir. Sabit ayarlı adaylar doğrulama average precision değeriyle karşılaştırılır. Her modelin alarm eşiği doğrulamada TSS = recall − yanlış pozitif oranını en çok artıran değerdir; eşitlikte daha az yanlış alarm seçilir. Seçim testten önce kaydedilir.

Olasılıklar modelin ham tahminidir; ayrıca kalibre edilmiş olduğu iddia edilmez. Kalibrasyon aşağıda ölçülür. Teste bakarak model veya eşik değiştirilmez.

In [ ]:
report = run_experiment(table, ROOT / "artifacts")
print(json.dumps(report["selection"], indent=2))
results = pd.DataFrame(report["metrics"])
display(results[["model", "split", "threshold", "recall", "precision", "fp", "fn", "false_positive_rate", "false_alarm_ratio", "average_precision", "brier", "tss"]])

## 5. Alarm dengesi ve kalibrasyon
Yanlış pozitif oranı = FP / (FP + TN). Yanlış alarm payı = FP / (TP + FP).
Alarm verilmediğinde precision ve yanlış alarm payı kodda 0 olarak raporlanır; alarm sayısıyla birlikte yorumlanmalıdır. Kalibrasyon noktalarının örnek sayısı azsa güvenilirlikleri düşüktür.

In [ ]:
chosen = report["selection"]["model"]
curves = pd.read_csv(ROOT / "artifacts/validation_thresholds.csv")
calibration = pd.read_csv(ROOT / "artifacts/calibration.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, curve in curves.groupby("model"):
    axes[0].plot(curve.false_alarms, curve.recall, label=name)
axes[0].set(xlabel="Yanlış alarm sayısı (doğrulama)", ylabel="Recall", title="Alarm dengesi")
axes[0].legend()
axes[1].plot([0, 1], [0, 1], "--", color="gray")
for name, group in calibration[calibration.split.eq("test")].groupby("model"):
    axes[1].plot(group.mean_probability, group.observed_rate, marker="o", label=name)
axes[1].set(xlabel="Ortalama tahmin", ylabel="Gerçek olay oranı", title="Test kalibrasyonu", xlim=(0,1), ylim=(0,1))
axes[1].legend()
plt.tight_layout()
plt.show()
display(calibration[(calibration.model == chosen) & (calibration.split == "test")])

## 6. Kaçırılan olaylar ve yanlış alarmlar
İyileştirme için hata örnekleri. Sonuçlar %10 örneklem için geçerlidir; örtüşen pencereler bağımsız değildir. Hatalara bakarak değiştirilen bir sonraki model bu test sonucunu yeni, dokunulmamış bir sonuç gibi sunmamalıdır.

In [ ]:
outcomes = pd.read_csv(ROOT / "artifacts/test_outcomes.csv")
display(outcomes.groupby("outcome").size().rename("windows").to_frame())
columns = ["harp", "forecast_time", "label", "probability", "outcome"]
display(outcomes[outcomes.outcome.eq("miss")][columns].head(10))
display(outcomes[outcomes.outcome.eq("false_alarm")][columns].sort_values("probability", ascending=False).head(10))
print("Panel: .venv/bin/streamlit run app.py")